Step 0 — Understand what you're packaging. Before writing a Dockerfile, make sure the API from the last challenge has two properties: the database URL comes from an environment variable, and dependencies are pinned in a requirements.txt (or lockfile). If you hardcoded localhost:5432 anywhere, fix that now — inside Docker, "localhost" means the container itself, and this single misunderstanding causes 80% of first-day Docker pain.

Step 1 — Write the naive Dockerfile first. Deliberately start with the simple version so the stretch goal has something to beat:

In [ ]:
FROM python:3.12
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

Build it (docker build -t myapi .), run it (docker run -p 8000:8000 myapi), and hit /docs. It will start and then fail to reach the database — that's expected and instructive; you've proven the app runs but has no friends yet. Two habits to establish here: --host 0.0.0.0 (uvicorn's default of 127.0.0.1 is unreachable from outside the container — the other classic day-one bug), and a .dockerignore file excluding .git, __pycache__, and your local venv, which shrinks build context and avoids leaking junk into the image.Build it (docker build -t myapi .), run it (docker run -p 8000:8000 myapi), and hit /docs. It will start and then fail to reach the database — that's expected and instructive; you've proven the app runs but has no friends yet. Two habits to establish here: --host 0.0.0.0 (uvicorn's default of 127.0.0.1 is unreachable from outside the container — the other classic day-one bug), and a .dockerignore file excluding .git, __pycache__, and your local venv, which shrinks build context and avoids leaking junk into the image.

Step 2 — Compose the app with Postgres. Now the payoff — one file that describes the whole system:

In [ ]:
services:
api:
build: .
ports: ["8000:8000"]
environment:
DATABASE_URL: postgresql://app:secret@db:5432/notes
depends_on:
db:
condition: service_healthy
db:
image: postgres:16
environment:
POSTGRES_USER: app
POSTGRES_PASSWORD: secret
POSTGRES_DB: notes
volumes:
- pgdata:/var/lib/postgresql/data
healthcheck:
test: ["CMD-READY", "pg_isready", "-U", "app"]
interval: 2s
retries: 10
volumes:
pgdata:

(Use ["CMD", "pg_isready", "-U", "app"] for the healthcheck test — and notice the hostname in DATABASE_URL is db, the service name; Compose gives you DNS between containers for free.) Three teaching moments hide in this file. The healthcheck plus condition: service_healthy exists because Postgres takes a few seconds to accept connections, and without it your API starts first and crashes — remove it once to watch that race happen. The named volume is what makes data survive docker compose down; test it by creating a note, tearing down, bringing it back up, and checking the note's still there. And docker compose logs api is your new best friend for debugging.

Step 3 — Prove the loop. The definition of done: a teammate clones the repo and runs exactly one command, docker compose up, and gets a working API with database on a machine that has nothing installed but Docker. In a mixed group, actually do this swap — everyone runs someone else's repo. Fresh-machine failures (a missing file that only existed locally, an unpinned dependency) are the whole lesson, and they only surface on someone else's laptop.

Stretch — multi-stage build under 200 MB. First, measure: docker images will show your naive image at roughly 1 GB. Now shrink it in ranked order of impact. The biggest win is the base image: switching FROM python:3.12 to python:3.12-slim drops ~700 MB on its own, because the full image carries compilers and dev headers you don't need at runtime. Then the multi-stage pattern — build wheels in one stage, copy only the installed packages into a clean slim stage:

In [ ]:
FROM python:3.12-slim AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --prefix=/install -r requirements.txt

FROM python:3.12-slim
COPY --from=builder /install /usr/local
WORKDIR /app
COPY . .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

This matters most when a dependency (like psycopg2 built from source) needs gcc to install — the compiler stays in the builder stage and never ships. Two refinements finish the job: order layers so COPY requirements.txt + pip install come before COPY . ., which means editing your code no longer triggers a dependency reinstall (watch rebuild time drop from minutes to seconds — layer caching is arguably a bigger daily win than image size), and add --no-cache-dir to pip. Getting under 200 MB with slim is very achievable; the ambitious can chase Alpine or distroless images, though be warned Alpine + Python has sharp edges (musl vs glibc wheels) that make for a good war story. One last production habit worth adding: create a non-root user (RUN useradd app + USER app) — running as root inside containers is the thing security reviews always flag.This matters most when a dependency (like psycopg2 built from source) needs gcc to install — the compiler stays in the builder stage and never ships. Two refinements finish the job: order layers so COPY requirements.txt + pip install come before COPY . ., which means editing your code no longer triggers a dependency reinstall (watch rebuild time drop from minutes to seconds — layer caching is arguably a bigger daily win than image size), and add --no-cache-dir to pip. Getting under 200 MB with slim is very achievable; the ambitious can chase Alpine or distroless images, though be warned Alpine + Python has sharp edges (musl vs glibc wheels) that make for a good war story. One last production habit worth adding: create a non-root user (RUN useradd app + USER app) — running as root inside containers is the thing security reviews always flag.